In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1] / "src"))

# Gross Code — LER vs Physical Error Rate (Importance Sampling)

Uses weight-stratified importance sampling (`importance_sampling.py`, based on arXiv:2511.15177) to estimate the logical error rate of the [[144,12,12]] gross code under three scenarios:

1. **Memory experiment** — bare BB syndrome rounds via `BBCodeSimulator(BB_144_12_12)`
2. **LPU X̄₁ circuit** — full gauging-measurement circuit for the X̄₁ logical operator
3. **LPU Z̄₁ circuit** — full gauging-measurement circuit for the Z̄₁ logical operator

**Decoder:** RelayBP (Rust-native) for all three.

### Why IS for the gross code?
Direct MC needs thousands of shots per p value to see any logical errors below threshold. The gross-code LPU circuits in particular run at ~3× the per-shot cost of the bare memory circuit, so a direct sweep over a 2-decade $p$ range is expensive. Importance sampling does one fixed-budget pass per circuit and gives the entire LER vs $p$ curve from a single reweighting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from bb_code_sim import BBCodeSimulator, BB_144_12_12, RelayBPDecoder
from surface_code_sim import ErrorModel
import gross_code_lpu_tdg as tdg
from importance_sampling import importance_sample

SEED = 42

# Common IS parameters
P_REF             = 0.005
# WEIGHTS must be a CONTIGUOUS block: the reweighting sum drops mass at any skipped
# weight. It has to bracket the failure onset (~ fault distance; heuristic d*ln(1/q)
# ~= 96 here) AND the dominant binomial mass mu(p)=N_expanded*q(p) across the grid.
# With the p-grid capped at ~1e-2, mu_max ~= 150, so 1..189 covers mu_max + a few sigma.
# (Run Section 0 to confirm the onset and trim the low end if desired.)
WEIGHTS           = list(range(1, 190))   # contiguous fault weights 1..189
SHOTS_PER_WEIGHT  = 150                   # samples per weight per circuit
# Cap near threshold: p above ~1e-2 is supra-threshold (LER ~ 0.5, uninteresting) and
# would push mu(p) up to ~471, forcing a much wider WEIGHTS block.
P_TARGETS         = np.logspace(-3.5, -2.0, 30)  # reweight to ~3.2e-4 .. 1e-2

# LPU parameters
C, D_INIT = 10, 12

print(f'Gross code: [[{BB_144_12_12.l*BB_144_12_12.m*2}, 12, {BB_144_12_12.distance}]]')
print(f'IS parameters: p_ref={P_REF}, weights={WEIGHTS[0]}..{WEIGHTS[-1]} '
      f'({len(WEIGHTS)} weights), shots_per_weight={SHOTS_PER_WEIGHT}')
print(f'Reweighting to {len(P_TARGETS)} target p values from {P_TARGETS[0]:.2e} to {P_TARGETS[-1]:.2e}')

## 0. Failure-onset scan — choose the sampling weights

`f(w)` is **zero for small `w`**: a weight-`w` fault set drawn from `N_expanded` mechanisms can only fail once `w` approaches the circuit's fault distance. Meanwhile the reweighting sum

$$P_\text{logical}(p)=\sum_w f(w)\,\binom{N}{w}\,q(p)^w\,(1-q(p))^{N-w}$$

is dominated by weights near the binomial mean $\mu(p)=N_\text{expanded}\cdot q(p)$. So `WEIGHTS` must bracket **both** the *onset* (where `f(w)` first exceeds 0) and the *dominant region* ($\mu(p)$ across the target grid). The original `WEIGHTS = 1..10` sat below both — which is why every reported `F/T` was 0.

This cell locates the onset empirically and reports the dominant-weight range, then suggests a `WEIGHTS` window. **Adjust `WEIGHTS` in the parameters cell above and re-run the sweeps below.**

> Per arXiv:2511.15177 the relevant weights scale with the code distance and $\log(1/q)$; the heuristic `d*ln(1/q_base)` is printed below as a cross-check, but the binomial mean $\mu$ is the exact center to anchor on. The scan decodes high-weight syndromes and may take a few minutes.

In [ ]:
from importance_sampling import _parse_dem, _expand, _sample_failures_at_weight


def dominant_weight_range(circuit, p_ref, p_values, q_base=None):
    """Exact center of the reweighting mass: mu(p) = N_expanded * q(p)."""
    probs, _, _ = _parse_dem(circuit)
    col_to_mech, qb, _ = _expand(probs, q_base)
    N = col_to_mech.shape[0]
    mu = N * qb * (np.asarray(p_values, float) / p_ref)
    return N, qb, mu


def failure_onset_scan(circuit, decoder, scan_weights, shots=50, q_base=None, seed=SEED):
    """Sample f(w) on a coarse weight grid to find where failures turn on.

    Uses the same weight-stratified sampler as importance_sample(), so the
    f(w) measured here is directly comparable to the full sweep.
    """
    rng = np.random.default_rng(seed)
    probs, det, obs = _parse_dem(circuit)
    col_to_mech, qb, _ = _expand(probs, q_base)
    decoder.setup(circuit)
    spectrum = {}
    for w in scan_weights:
        F = _sample_failures_at_weight(det, obs, col_to_mech, w, shots, decoder, rng)
        spectrum[w] = F / shots
        print(f'  w={w:>4}: F/T = {F:>3}/{shots} = {F / shots:.3f}')
    return spectrum


# --- run on the (cheapest) memory circuit to size WEIGHTS ---
em_ref = ErrorModel.symmetric(P_REF)
scan_circuit = BBCodeSimulator(BB_144_12_12).build_circuit(em_ref, rounds=BB_144_12_12.distance)

d = BB_144_12_12.distance
N_exp, q_base, mu = dominant_weight_range(scan_circuit, P_REF, P_TARGETS)
print(f'N_expanded={N_exp}, q_base={q_base:.3e}')
print(f'dominant weight mu(p)=N*q over target grid: '
      f'{mu.min():.1f} (p={P_TARGETS[0]:.1e}) .. {mu.max():.1f} (p={P_TARGETS[-1]:.1e})')
print(f'paper heuristic critical weight ~ d*ln(1/q_base) = {d * np.log(1 / q_base):.0f}  '
      f'(cross-check vs arXiv:2511.15177)')
print()

# coarse grid: fine near the distance (onset), coarse up to ~2x the mean at p_ref
w_hi = int(round(2 * N_exp * q_base))
scan_weights = sorted(set(
    list(range(2, 2 * d + 1, 2)) +
    [int(round(x)) for x in np.linspace(2 * d, w_hi, 8)]
))
print(f'Scanning {len(scan_weights)} weights: {scan_weights}')
onset = failure_onset_scan(scan_circuit, RelayBPDecoder(), scan_weights, shots=50)

nz = [w for w, f in onset.items() if f > 0]
if nz:
    print(f'\nFailures first appear at w={nz[0]}; suggest WEIGHTS spanning ~{nz[0]} .. {w_hi}')
else:
    print('\nNo failures in scan range — increase the max scan weight or shots.')

## 1. Memory experiment — IS sweep

Build one Z-memory circuit at `p_ref`, run IS, reweight to the full $p$ grid.

In [ ]:
em_ref = ErrorModel.symmetric(P_REF)

sim_mem = BBCodeSimulator(BB_144_12_12)
mem_circuit = sim_mem.build_circuit(em_ref, rounds=BB_144_12_12.distance)
print(f'Memory circuit: {mem_circuit.num_qubits} qubits, {mem_circuit.num_detectors} detectors')

t0 = time.perf_counter()
mem_is = importance_sample(
    mem_circuit, RelayBPDecoder(),
    p_ref=P_REF, p_values=P_TARGETS,
    weights=WEIGHTS, shots_per_weight=SHOTS_PER_WEIGHT,
    seed=SEED,
)
print(f'Memory IS done in {time.perf_counter()-t0:.1f}s')
print(f'  N_expanded={mem_is.spectrum.n_expanded}, q_base={mem_is.spectrum.q_base:.5f}')
print()
print('  Failure spectrum:')
for w, T, F in zip(mem_is.spectrum.weights, mem_is.spectrum.trials, mem_is.spectrum.failures):
    print(f'    w={w:>2}: F/T = {F:>4}/{T}  =  {F/T:.3f}')

## 2. LPU X̄₁ circuit — IS sweep

In [ ]:
x1_circuit = tdg.build_logical_x1_circuit(em_ref, C=C, d_init=D_INIT)
print(f'X̄₁ LPU circuit: {x1_circuit.num_qubits} qubits, {x1_circuit.num_detectors} detectors')

t0 = time.perf_counter()
x1_is = importance_sample(
    x1_circuit, RelayBPDecoder(),
    p_ref=P_REF, p_values=P_TARGETS,
    weights=WEIGHTS, shots_per_weight=SHOTS_PER_WEIGHT,
    seed=SEED,
)
print(f'X̄₁ IS done in {time.perf_counter()-t0:.1f}s')
print(f'  N_expanded={x1_is.spectrum.n_expanded}, q_base={x1_is.spectrum.q_base:.5f}')
print()
print('  Failure spectrum:')
for w, T, F in zip(x1_is.spectrum.weights, x1_is.spectrum.trials, x1_is.spectrum.failures):
    print(f'    w={w:>2}: F/T = {F:>4}/{T}  =  {F/T:.3f}')

## 3. LPU Z̄₁ circuit — IS sweep

In [ ]:
z1_circuit = tdg.build_logical_z1_circuit(em_ref, C=C, d_init=D_INIT)
print(f'Z̄₁ LPU circuit: {z1_circuit.num_qubits} qubits, {z1_circuit.num_detectors} detectors')

t0 = time.perf_counter()
z1_is = importance_sample(
    z1_circuit, RelayBPDecoder(),
    p_ref=P_REF, p_values=P_TARGETS,
    weights=WEIGHTS, shots_per_weight=SHOTS_PER_WEIGHT,
    seed=SEED,
)
print(f'Z̄₁ IS done in {time.perf_counter()-t0:.1f}s')
print(f'  N_expanded={z1_is.spectrum.n_expanded}, q_base={z1_is.spectrum.q_base:.5f}')
print()
print('  Failure spectrum:')
for w, T, F in zip(z1_is.spectrum.weights, z1_is.spectrum.trials, z1_is.spectrum.failures):
    print(f'    w={w:>2}: F/T = {F:>4}/{T}  =  {F/T:.3f}')

## 4. Comparison plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for result, label, color in [
    (mem_is, 'Memory (d=12 rounds)',          'steelblue'),
    (x1_is,  f'LPU X̄₁ (C={C}, d_init={D_INIT})', 'tomato'),
    (z1_is,  f'LPU Z̄₁ (C={C}, d_init={D_INIT})', 'seagreen'),
]:
    lo = np.maximum(result.P_logical - result.P_logical_se, 1e-15)
    hi = result.P_logical + result.P_logical_se
    ax.fill_between(result.p_values, lo, hi, color=color, alpha=0.25)
    ax.plot(result.p_values, result.P_logical, '-', color=color, label=label, lw=2)

ax.axhline(0.5, color='grey', ls='--', lw=0.8, label='50% (random)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate (importance-sampled)')
ax.set_title(f'[[144,12,12]] Gross Code — LER vs $p$ via IS  (RelayBP decoder)\n'
             f'{len(WEIGHTS)} weights × {SHOTS_PER_WEIGHT} shots/circuit, reweighted to {len(P_TARGETS)} p points')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. LER ratios — LPU overhead at fixed $p$

The LPU circuits do more work than the bare memory circuit (extra rounds, edge qubits). At any given $p$, how much LER does that cost?

In [ ]:
ratio_x1 = x1_is.P_logical / np.maximum(mem_is.P_logical, 1e-15)
ratio_z1 = z1_is.P_logical / np.maximum(mem_is.P_logical, 1e-15)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(P_TARGETS, ratio_x1, 'o-', color='tomato',   label='X̄₁ / memory', lw=2)
ax.plot(P_TARGETS, ratio_z1, 's-', color='seagreen', label='Z̄₁ / memory', lw=2)
ax.axhline(1.0, color='grey', ls='--', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('LPU LER / memory LER')
ax.set_title('LPU overhead relative to bare memory experiment')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Failure-spectrum ansatz (Technique I) — extrapolated LER

Fit the smooth ansatz of arXiv:2511.15177 §3 (Eq. 10) to each sampled failure spectrum and reweight the **fitted** $f(w)$ over all weights $0..N$. Unlike the direct reweighting in §1–4, this does **not** require `WEIGHTS` to bracket the full dominant mass $\mu(p)=N\,q(p)$ — the fit fills in the tail, so a sparse `WEIGHTS` set suffices, and $P(p)$ can be extrapolated to low $p$. The saturation $a = 1 - 2^{-K}$ is read from the number of logical observables $K$.

In [ ]:
from importance_sampling import fit_failure_spectrum, logical_error_rate_from_ansatz

K = mem_circuit.detector_error_model(decompose_errors=False).num_observables
print(f'K = {K} logical observables  ->  saturation a = {1 - 2.0**-K:.6f}')

ANSATZ_MODEL = 'f3'   # 'f2' / 'f3' / 'f5' — more params fit better but need more nonzero points

fits = {}
for name, res in [('Memory', mem_is), ('X̅1', x1_is), ('Z̅1', z1_is)]:
    fit = fit_failure_spectrum(res.spectrum, K=K, model=ANSATZ_MODEL)
    P_ext = logical_error_rate_from_ansatz(fit, P_TARGETS)
    fits[name] = (fit, P_ext)
    print(f'  {name:7s}: {fit.model}  ' + ', '.join(f'{k}={v:.3g}' for k, v in fit.params.items())
          + f'   (n_points={fit.n_points})')

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = {'Memory': 'steelblue', 'X̅1': 'tomato', 'Z̅1': 'seagreen'}
res_by_name = {'Memory': mem_is, 'X̅1': x1_is, 'Z̅1': z1_is}
for name, (fit, P_ext) in fits.items():
    res = res_by_name[name]
    c = colors[name]
    ax.plot(res.p_values, res.P_logical, 'o', color=c, alpha=0.5, ms=4, label=f'{name} (reweighted)')
    ax.plot(P_TARGETS, P_ext, '-', color=c, lw=2, label=f'{name} (ansatz {fit.model})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$'); ax.set_ylabel('Logical error rate')
ax.set_title(f'[[144,12,12]] — ansatz-extrapolated LER vs direct reweighting ({ANSATZ_MODEL})')
ax.legend(fontsize=8, ncol=2); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Min-weight onset (Technique II)

Compute the circuit fault distance $D$ and the optimal onset $w_0^* = \lceil D/2 \rceil$ (arXiv:2511.15177 §4.1, via BP-OSD on the per-logical extended check matrix), then use the onset to **anchor** the ansatz fit (`w0=dr.onset`). This pins the low-$w$ behaviour the ansatz extrapolates from, instead of leaving $w_0$ as a free fit parameter.

In [ ]:
from min_weight import compute_distance

# Distance + optimal onset from the (cheapest) memory circuit.
# NOTE: compute_distance runs BP-OSD per logical and may take a few minutes on the
# gross circuit; for large codes it returns an UPPER BOUND on D.
dr = compute_distance(mem_circuit, osd_order=10, max_iter=200)
print(f'circuit fault distance D = {dr.distance}  (upper bound for large codes)')
print(f'optimal onset w0* = ceil(D/2) = {dr.onset}')

# Re-fit the memory spectrum with the onset pinned as an exact anchor:
fit_anchored = fit_failure_spectrum(mem_is.spectrum, K=K, model=ANSATZ_MODEL, w0=dr.onset)
print('anchored fit params:', {k: round(v, 4) for k, v in fit_anchored.params.items()})

# The exact onset fraction f*(D/2) (min_weight.optimal_onset_fraction) is intractable
# for the full [[144,12,12]] circuit (paper Table 2: ~1e23 fails). Use it on small /
# bit-flip systems, or the sampling estimator for large ones.